# Refactoring Code & Unit Testing

### Loading Libraries

In [ ]:
# Numerical Computing
import math
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualisation
import matplotlib
import seaborn as sns
import matplotlib.pyplot as plt

# SciPy
from scipy.ndimage import shift

# YahooFinance
import yfinance as yf

# PyTest
import pytest

In [ ]:
# port_tickers = [
#     'QCOM', 'TSLA', 'NFLX', 'DIS', 'PG',
#     'MMM', 'IBM', 'BRK-B', 'UPS', 'F'
# ]

# bm_ticker = '^GSPC'

# ticker_list = [bm_ticker] + port_tickers

# period = '6mo'

# price_df_rf = get_tickers(
#     ticker_list=ticker_list,
#     period=period
# )

# price_df_rf.to_csv('data/tickers-raw.csv')

In [ ]:
%%writefile test_pd_refactor.py

@pytest.fixture
def raw_price_df():
    return pd.read_csv(
        'data/tickers-raw.csv',
        index_col='Date',
        parse_dates=['Date'],
        dtype_backend='pyarrow',
        engine='pyarrow'
    )


def test_basic(raw_price_df):
    assert len(raw_price_df) > 1

In [ ]:
%%writefile returns.py

def np_returns(df, col):
    values = df[col].to_numpy()

    shifted = shift(values, 1, cval=np.NaN)

    res = np.round(
        np.subtract(
            np.exp(
                np.nancumsum(
                    np.log(
                        np.divide(values, shifted)
                    )
                )
            ),
            1
        ),
        3
    )

    res[0] = np.NaN

    return pd.Series(res, index=df.index)

In [ ]:
def get_tickers(
    ticker_list,
    period,
    interval='1d'
):
    return (
        pd.read_csv(
            'data/raw-yfinance.csv',
            index_col=0,
            header=[0, 1]
        )['Close']
        .round(2)
    )

In [ ]:
def get_portfolio(
    port_tickers,
    price_df,
    benchmark_data
):

    df_data = {
        'Beta': [
            1.34, 2, 0.75, 1.2, 0.41,
            0.95, 1.23, 0.9, 1.05, 1.15
        ],

        'Shares': [
            -1900, -100, -400, -800,
            -5500, 1600, 1800, 2800,
            1100, 20800
        ],

        'rSL': [
            42.75, 231, 156, 54.2,
            37.5, 42.75, 29.97,
            59.97, 39.97, 2.10
        ]
    }

    benchmark_cost = benchmark_data.iloc[0]
    benchmark_price = benchmark_data.iloc[-1]

    start_price = price_df.iloc[0]
    end_price = price_df.iloc[-1]

    return (
        pd.DataFrame(
            df_data,
            index=port_tickers
        )
        .assign(
            Side=lambda df_:
                np.sign(df_.Shares),

            rCost=(
                (start_price / benchmark_cost)
                .mul(1000)
                .round(2)
            ),

            rPrice=(
                (end_price / benchmark_price)
                .mul(1000)
                .round(2)
            ),

            Cost=start_price,
            Price=end_price,
        )
    )